# Qwen3-ASR Speech Recognition with OpenVINO™

The Qwen3-ASR family includes Qwen3-ASR-1.7B and Qwen3-ASR-0.6B, which support language identification and ASR for 52 languages and dialects. Both leverage large-scale speech training data and the strong audio understanding capability of their foundation model, Qwen3-Omni.

* **All-in-one**: language identification and speech recognition for many languages, including English accents from multiple countries and regions.
* **Excellent and Fast**: high-quality, robust recognition; the 0.6B version offers an accuracy-efficiency trade-off and supports long audio.
* **Forced alignment**: `Qwen3-ForcedAligner-0.6B` predicts word-level timestamps in 11 languages.

In this tutorial we run and optimize **Qwen3-ASR** and the **Qwen3-ForcedAligner** with OpenVINO using [Optimum Intel](https://huggingface.co/docs/optimum/intel/index). Usage mirrors the original Hugging Face model cards.

More details: original [repository](https://github.com/QwenLM/Qwen3-ASR) and [model card](https://huggingface.co/Qwen/Qwen3-ASR-0.6B-hf).

#### Table of contents:
- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert model to OpenVINO IR](#Convert-model-to-OpenVINO-IR)
- [Select inference device](#Select-inference-device)
- [Run speech recognition](#Run-speech-recognition)
- [Word-level timestamps with the forced aligner](#Word-level-timestamps-with-the-forced-aligner)
- [Interactive demo](#Interactive-demo)


## Prerequisites
[back to top ⬆️](#Table-of-contents:)


In [ ]:
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-asr.ipynb")


### Install dependencies

The HuggingFace-native Qwen3-ASR models require `transformers>=5` and the Qwen3-ASR support in Optimum Intel. We install Optimum Intel from the feature branch that adds it.


In [ ]:
from pip_helper import pip_install

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch>=2.6",
    "torchaudio",
    "librosa",
    "soundfile",
    "gradio>=4.19",
    "scipy",
    "nncf>=2.19.0",
    "openvino>=2025.4.0",
    "openvino-tokenizers>=2025.4.0",
    "transformers>=5.0.0",
)

# Optimum Intel with HuggingFace-native Qwen3-ASR / Qwen3-ForcedAligner support
pip_install("-q", "git+https://github.com/openvino-dev-samples/optimum-intel.git@add-qwen3-asr-hf-and-forced-aligner")


## Select model
[back to top ⬆️](#Table-of-contents:)

Select the Qwen3-ASR variant. The 0.6B model is recommended for faster inference.


In [ ]:
import ipywidgets as widgets

model_ids = [
    "Qwen/Qwen3-ASR-0.6B-hf",
    "Qwen/Qwen3-ASR-1.7B-hf",
]

model_selector = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
)

model_selector


## Convert model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

Optimum Intel exposes the same model API as 🤗 Transformers, with an OpenVINO backend. `OVModelForSpeechSeq2Seq` exports the model to OpenVINO IR on the fly (`export=True`) and splits it into an audio encoder and a text decoder.

This is equivalent to the CLI export:

```bash
optimum-cli export openvino --model Qwen/Qwen3-ASR-0.6B-hf --task automatic-speech-recognition Qwen3-ASR-0.6B-hf-ov
```

You can also compress the model weights to INT8 by passing `--weight-format int8` to the CLI, or a `quantization_config` / `load_in_8bit=True` to `from_pretrained`.


In [ ]:
from optimum.intel import OVModelForSpeechSeq2Seq

model_id = model_selector.value
model_name = model_id.split("/")[-1]
ov_model_dir = Path(f"{model_name}-ov")

# Set to True to compress the model weights to INT8
load_in_8bit = False

if not ov_model_dir.exists():
    ov_model = OVModelForSpeechSeq2Seq.from_pretrained(model_id, export=True, load_in_8bit=load_in_8bit)
    ov_model.save_pretrained(ov_model_dir)
    del ov_model
    print(f"Model exported to {ov_model_dir}")
else:
    print(f"Model already exported to {ov_model_dir}")


## Select inference device
[back to top ⬆️](#Table-of-contents:)


In [ ]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])

device


## Run speech recognition
[back to top ⬆️](#Table-of-contents:)

We load the OpenVINO model and the processor and run transcription. Usage is the same as the original model: build the inputs with `processor.apply_transcription_request`, call `generate`, and decode with `processor.decode`.


In [ ]:
import librosa
from transformers import AutoProcessor
from optimum.intel import OVModelForSpeechSeq2Seq

processor = AutoProcessor.from_pretrained(model_id)
ov_model = OVModelForSpeechSeq2Seq.from_pretrained(ov_model_dir, device=device.value)


In [ ]:
import urllib.request

sample_audio = Path("asr_en.wav")
if not sample_audio.exists():
    urllib.request.urlretrieve(
        "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav", sample_audio
    )

audio, sr = librosa.load(sample_audio, sr=16000)

inputs = processor.apply_transcription_request(audio=audio, sampling_rate=16000)
inputs = inputs.to(ov_model.device)

generated_ids = ov_model.generate(**inputs, max_new_tokens=256)
generated_ids = generated_ids[:, inputs["input_ids"].shape[1]:]

parsed = processor.decode(generated_ids, return_format="parsed")[0]
print(f"Detected language: {parsed['language']}")
print(f"Transcription: {parsed['transcription']}")


You can pass a `language` hint (e.g. `language="English"`) to `apply_transcription_request` to skip auto-detection.


## Word-level timestamps with the forced aligner
[back to top ⬆️](#Table-of-contents:)

`Qwen3-ForcedAligner-0.6B` predicts word-level timestamps for a known transcript. With Optimum Intel it is exposed as `OVModelForQwen3ASRForcedAligner`, used exactly like the original `Qwen3ASRForTokenClassification`: build inputs with `processor.prepare_forced_aligner_inputs`, run a single forward pass, and decode with `processor.decode_forced_alignment`.


In [ ]:
from transformers import AutoProcessor
from optimum.intel import OVModelForQwen3ASRForcedAligner

aligner_id = "Qwen/Qwen3-ForcedAligner-0.6B-hf"
aligner_dir = Path("Qwen3-ForcedAligner-0.6B-hf-ov")

if not aligner_dir.exists():
    aligner_model = OVModelForQwen3ASRForcedAligner.from_pretrained(aligner_id, export=True)
    aligner_model.save_pretrained(aligner_dir)
    del aligner_model

aligner_processor = AutoProcessor.from_pretrained(aligner_id)
aligner_model = OVModelForQwen3ASRForcedAligner.from_pretrained(aligner_dir, device=device.value)


In [ ]:
import torch

transcript = parsed["transcription"]
language = parsed["language"]

aligner_inputs, word_lists = aligner_processor.prepare_forced_aligner_inputs(
    audio=audio,
    transcript=transcript,
    language=language,
)
aligner_inputs = aligner_inputs.to(aligner_model.device)

with torch.inference_mode():
    outputs = aligner_model(**aligner_inputs)

timestamps = aligner_processor.decode_forced_alignment(
    logits=outputs.logits,
    input_ids=aligner_inputs["input_ids"],
    word_lists=word_lists,
    timestamp_token_id=aligner_model.config.timestamp_token_id,
)[0]

for item in timestamps:
    print(f"{item['start_time']:6.2f}s - {item['end_time']:6.2f}s  {item['text']}")


## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Launch a Gradio demo to upload or record audio, transcribe it, and view word-level timestamps.


In [ ]:
from gradio_helper import make_demo

demo = make_demo(asr_model=ov_model, processor=processor, aligner_model=aligner_model)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# If you are launching remotely, specify server_name and server_port:
#   demo.launch(server_name='your_server_name', server_port=7860)
